# Embedding and RMSNorm demo

(Notebook version of `scripts/embedding_norm_demo.py`.)

The first two pieces of the transformer:

1. **Embedding.** Each token id is replaced by a learned vector of `hidden_size` numbers. This is a plain
   table lookup, with no arithmetic.
2. **RMSNorm.** Rescales each token's vector so its typical size (root-mean-square) is 1. It keeps training
   stable, because every layer sees inputs of a similar scale. Only the length of the vector changes, not its
   direction.

We use the `nano` config with random weights, so the actual numbers mean nothing yet. The point is the
shapes and the behaviour.

Select the kernel **smalllm (.venv)** before running.

In [1]:
from pathlib import Path

import torch

from smalllm.model import ModelConfig, RMSNorm, Transformer

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
IDS = [6403, 1980, 253, 655]  # "Once upon a time" (see tokenize_demo.ipynb)

torch.manual_seed(0)
cfg = ModelConfig.from_yaml(ROOT / "configs" / "nano.yaml")
model = Transformer(cfg)
ids = torch.tensor([IDS])  # shape (1, 4): a batch of one sentence of four tokens


def rms(x: torch.Tensor) -> list[float]:
    """Root-mean-square (typical size) of each token vector, for a (1, T, hidden) tensor."""
    return [round(v, 4) for v in x.pow(2).mean(-1).sqrt()[0].tolist()]

## 1. Embedding
The table has one row per vocabulary entry. Looking up four ids gives four vectors.

In [2]:
table = model.model.embed_tokens
print(f"table shape:  {tuple(table.weight.shape)}  (vocab, hidden)")

vectors = table(ids)
print(f"ids shape:    {tuple(ids.shape)}  ->  vectors shape: {tuple(vectors.shape)}")

table shape:  (49152, 128)  (vocab, hidden)
ids shape:    (1, 4)  ->  vectors shape: (1, 4, 128)


An embedding is a pure row lookup. Token 6403 just gets row 6403 of the table.

In [3]:
print(f"is row {IDS[0]} of the table: {torch.equal(vectors[0, 0], table.weight[IDS[0]])}")
print(f"first 6 numbers of token {IDS[0]}: {[round(v, 4) for v in vectors[0, 0, :6].tolist()]}")
print(f"rms of each token vector: {rms(vectors)}")

is row 6403 of the table: True
first 6 numbers of token 6403: [-0.0428, -0.0068, 0.0075, -0.0007, -0.0271, -0.0145]
rms of each token vector: [0.0204, 0.0196, 0.0188, 0.0204]


## 2. RMSNorm on tokens of very different scale
We multiply the four vectors by 1, 10, 100 and 0.01, then normalise. All come out near rms 1. The last token
is so small that `eps` (a tiny constant that prevents dividing by zero) dominates, so it stays small.

In [4]:
norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)

scales = torch.tensor([1.0, 10.0, 100.0, 0.01]).view(1, 4, 1)
scaled = vectors * scales
normed = norm(scaled)
print(f"rms before: {rms(scaled)}")
print(f"rms after:  {rms(normed)}")

rms before: [0.0204, 0.196, 1.8775, 0.0002]
rms after:  [0.9882, 0.9999, 1.0, 0.0643]


Check against the formula written by hand, `x * rsqrt(mean(x²) + eps)`. The learned weight starts at 1.0.

In [5]:
by_hand = scaled * torch.rsqrt(scaled.pow(2).mean(-1, keepdim=True) + cfg.rms_norm_eps)
print(f"matches the formula by hand: {torch.allclose(by_hand, normed)}")
print(f"initial norm weights (first 4): {norm.weight[:4].tolist()}")

matches the formula by hand: True
initial norm weights (first 4): [1.0, 1.0, 1.0, 1.0]


Only the length changes, not the direction: the cosine similarity before and after is 1.

In [6]:
cosine = torch.cosine_similarity(scaled[0, 0], normed[0, 0], dim=0).item()
print(f"cosine similarity before/after for token 0: {cosine:.4f}")

cosine similarity before/after for token 0: 1.0000


## 3. RMSNorm step by step on one real vector
The same operation split into its individual arithmetic steps, on token 0's `hidden_size` numbers.

In [7]:
x = vectors[0, 0].detach()  # shape (hidden,)
h = cfg.hidden_size


def first6(t: torch.Tensor) -> list[float]:
    return [round(v, 4) for v in t[:6].tolist()]


print(f"x, first 6: {first6(x)}")

x, first 6: [-0.0428, -0.0068, 0.0075, -0.0007, -0.0271, -0.0145]


**Steps 1–2:** square every number, then take the mean.

In [8]:
squares = x.pow(2)
print(f"1. squared, first 6:       {[f'{v:.6f}' for v in squares[:6].tolist()]}")
print(f"   sum of all {h} squares:  {squares.sum().item():.6f}")

mean_square = squares.mean()
print(f"2. mean square (sum / {h}): {mean_square.item():.8f}")

1. squared, first 6:       ['0.001831', '0.000046', '0.000056', '0.000000', '0.000734', '0.000210']
   sum of all 128 squares:  0.053352
2. mean square (sum / 128): 0.00041681


**Steps 3–4:** add `eps`, take the reciprocal square root to get a scale factor, and multiply every number by it.

In [9]:
with_eps = mean_square + cfg.rms_norm_eps
print(f"3. + eps:                  {with_eps.item():.8f}")
print(f"   rms = sqrt of that:     {with_eps.sqrt().item():.6f}")

scale = torch.rsqrt(with_eps)
print(f"4. scale = 1 / rms:        {scale.item():.4f}")
y = x * scale
print(f"   x * scale, first 6:     {first6(y)}")
print(f"   y / x, first 6:         {first6(y / x)}  (the same factor for every number)")

3. + eps:                  0.00042681
   rms = sqrt of that:     0.020659
4. scale = 1 / rms:        48.4039
   x * scale, first 6:     [-2.0711, -0.33, 0.3625, -0.0332, -1.3114, -0.7021]
   y / x, first 6:         [48.4039, 48.4039, 48.4039, 48.4039, 48.4039, 48.4039]  (the same factor for every number)


**Before vs after:** the rms becomes 1, and the Euclidean length becomes √hidden.

In [10]:
print(f"rms:              {squares.mean().sqrt().item():.4f} -> {y.pow(2).mean().sqrt().item():.4f}")
print(f"largest |value|:  {x.abs().max().item():.4f} -> {y.abs().max().item():.4f}")
print(f"euclidean length: {x.norm().item():.4f} -> {y.norm().item():.4f}  (sqrt({h}) = {h**0.5:.4f})")

module_out = RMSNorm(h, cfg.rms_norm_eps)(x)
print(f"RMSNorm module gives the same numbers: {torch.allclose(module_out, y, atol=1e-6)}")

rms:              0.0204 -> 0.9882
largest |value|:  0.0566 -> 2.7387
euclidean length: 0.2310 -> 11.1804  (sqrt(128) = 11.3137)
RMSNorm module gives the same numbers: True
